# Data Wrangling Task 7D: Analysis of the NHANES 2021-2023 Cycle
* **Dheisitha Kulunuguna**
* **Undergraduate/SIT220**

## Introduction

This notebook presents an exploratory analysis of selected datasets from the 2021–2023 NHANES cycle, focusing on the relationships between demographic characteristics, anthropometric measurements, cardiovascular indicators, and smoking behaviour. Five NHANES datasets (DEMO, BMX, BPXO, TCHOL, and SMQ), were merged to form a unified analytical dataset. After cleaning, transforming, and deriving key variables such as BMI category, hypertensive status, and cholesterol classification, a series of interactive Bokeh visualizations were created to uncover meaningful patterns within the population.

The aim of this analysis is to identify interesting associations between lifestyle factors and cardiometabolic health while demonstrating a broad range of data wrangling techniques and the use of interactive, reader-controlled charts and tables. The notebook also briefly reflects on the ethical considerations involved in handling human health data, even when fully de-identified, ensuring the analysis remains both statistically informative and ethically responsible.

## Importing and Loading the Data

In [ ]:
import numpy as np
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Slider, Select, HoverTool, CDSView, CustomJS, RangeSlider, DataTable, TableColumn, TextInput, Tabs, TabPanel, FactorRange
from bokeh.layouts import row, column
from bokeh.io import output_notebook, curdoc
from bokeh.transform import factor_cmap, jitter

In [ ]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
output_notebook()
curdoc().theme = "dark_minimal"

In [ ]:
bmi_df = pd.read_sas("BMX_L.xpt")
bp_df = pd.read_sas("BPXO_L.xpt")
demo_df = pd.read_sas("DEMO_L.xpt")
smoke_df = pd.read_sas("SMQ_L.xpt")
chol_df = pd.read_sas("TCHOL_L.xpt")

## Merging the Dataframes

In [ ]:
merged = demo_df.merge(bmi_df, on="SEQN", how="left") \
  .merge(bp_df, on="SEQN", how="left") \
  .merge(smoke_df, on = "SEQN", how = "left") \
  .merge(chol_df, on = "SEQN", how = "left")

In [ ]:
merged

## Cleaning the Data

We will begin by checking for any duplicate rows and dropping them, if any.

In [ ]:
merged.duplicated().sum()

In [ ]:
merged["RIDRETH3"] = merged["RIDRETH3"].replace({
    7: "Other Race - Including Multi-Racial"
})

Next, we will convert values such as "Don't know" and "Refused" to NaN.

In [ ]:
special_missing = [7, 9, 77, 99, 777, 999]
merged = merged.replace(special_missing, np.nan)

Afterwards, we will be dropping irrelevant columns and columns with too many missing values.

In [ ]:
merged.isna().sum()

In [ ]:
drop_cols = [
    "RIDEXAGM", "DMDYRUSR", "RIDEXPRG", "DMDHRGND", "DMDHRAGZ", "RIDEXMON",
    "DMDBORN4", "DMDHHSIZ", "BMDSTATS",
    "DMDHREDZ", "DMDHRMAZ", "DMDHSEDZ", "SMD630", "SMQ621", "RIDRETH1",
    "SMD641", "SMD100MN", "SMD650", "BMIHIP", "BMIWAIST", "SDDSRVYR",
    "BMIARMC", "BMIARML", "BMILEG", "BMDBMIC", "BMIHT", "BPAOCSZ",
    "BMIHEAD", "BMXHEAD", "BMIRECUM", "BMXRECUM", "BMIWT", "RIDAGEMN"
    ]
merged = merged.drop(columns = drop_cols)

In [ ]:
merged.head()

In [ ]:
merged.isna().sum()

Next, we will be renaming columns for readability.

In [ ]:
df = merged.rename(columns = {
    "RIDSTATR": "interview_or_exam",
    "RIAGENDR": "gender",
    "RIDAGEYR": "age_years",
    "RIDRETH3": "race_ethnicity",
    "DMQMILIZ": "served_us_armed_forces",
    "DMDEDUC2": "education_level",
    "DMDMARTZ": "marital_status",
    "WTINT2YR": "weight_interview",
    "WTMEC2YR": "weight_mec",
    "SDMVSTRA": "variance_pseudo_stratum",
    "SDMVPSU": "variance_pseudo_psu",
    "INDFMPIR": "fam_income_to_poverty",

    "BMXWT": "weight_kg",
    "BMXHT": "height_cm",
    "BMXBMI": "bmi",
    "BMXLEG": "leg_cm",
    "BMXARML": "arm_cm",
    "BMXARMC": "arm_circ_cm",
    "BMXWAIST": "waist_cm",
    "BMXHIP": "hip_cm",

    "BPAOARM": "arm_selected",
    "BPAOABP": "abdomen_bp",
    "BPXOSY1": "systolic_bp_1",
    "BPXODI1": "diastolic_bp_1",
    "BPXOSY2": "systolic_bp_2",
    "BPXODI2": "diastolic_bp_2",
    "BPXOSY3": "systolic_bp_3",
    "BPXODI3": "diastolic_bp_3",
    "BPXOPLS1": "pulse_1",
    "BPXOPLS2": "pulse_2",
    "BPXOPLS3": "pulse_3",

    "SMQ020": "smoked_100_life",
    "SMQ040": "smoke_now",
    "SMAQUEX2": "questionnaire_mode",

    "LBDTCSI": "cholesterol_mmol_l",
    "LBXTC": "cholesterol_mg_dl",
    "WTPH2YR": "phleb_weight"
    })

In [ ]:
df

We will now be creating some derived variables.

In [ ]:
df["age_group"] = pd.cut(
    df["age_years"],
    bins=[0, 18, 39, 59, 150],
    labels=["0–18", "19–39", "40–59", "60+"],
)

In [ ]:
df["height_m"] = df["height_cm"] / 100.0

if "bmi" not in df or df["bmi"].isna().sum() > 0:
    df["bmi_computed"] = df["weight_kg"] / (df["height_m"] ** 2)
    df["bmi"] = df["bmi"].fillna(df["bmi_computed"])

df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
)

In [ ]:
sys_cols = ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
dias_cols = ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]
pulse_cols = ["pulse_1", "pulse_2", "pulse_3"]

df["bp_sys_mean"] = df[sys_cols].mean(axis=1)
df["bp_dias_mean"] = df[dias_cols].mean(axis=1)
df["pulse_mean"] = df[pulse_cols].mean(axis=1)

# Hypertension status
df["hypertensive"] = np.where((df["bp_sys_mean"] >= 130) | (df["bp_dias_mean"] >= 80), "Yes", "No")

In [ ]:
df["cholesterol_category"] = pd.cut(
    df["cholesterol_mg_dl"],
    bins=[0, 200, 240, 500],
    labels=["Desirable (<200)", "Borderline (200–239)", "High (≥240)"]
)

In [ ]:
def classify_smoker(row):
    ever_100 = row["smoked_100_life"]
    now = row["smoke_now"]

    if pd.isna(ever_100):
        return np.nan

    if ever_100 == 1 and now in [1, 2]:
        return "Current smoker"

    if ever_100 == 1 and now == 3:
        return "Former smoker"

    if ever_100 == 2:
        return "Never smoker"

    return np.nan

df["smoking_status"] = df.apply(classify_smoker, axis=1)

In [ ]:
df

Finally, we will be replacing the encoded values with their real values and dropping some redundant columns.

In [ ]:
df["interview_or_exam"] = df["interview_or_exam"].replace({
    1: "Interviewed only",
    2: "Both interviewed and MEC examined"
})
df["gender"] = df["gender"].replace({
    1: "Male",
    2: "Female"
})
df["race_ethnicity"] = df["race_ethnicity"].replace({
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian"
})
df["served_us_armed_forces"] = df["served_us_armed_forces"].replace({
    1: "Yes",
    2: "No"
})
df["education_level"] = df["education_level"].replace({
    1: "Less than 9th grade",
    2: "9-11th grade (Includes 12th grade with no diploma)",
    3: "High school graduate/GED or equivalent",
    4: "Some college or AA degree",
    5: "College graduate or above"
})
df["marital_status"] = df["marital_status"].replace({
    1: "Married/Living with partner",
    2: "Widowed/Divorced/Separated",
    3: "Never married"
})
df["arm_selected"] = df["smoked_100_life"].replace({
    1: "Yes",
    2: "No"
})
df["smoked_100_life"] = df["smoked_100_life"].replace({
    1: "Yes",
    2: "No"
})
df["smoke_now"] = df["smoke_now"].replace({
    1: "Everyday",
    2: "Some days",
    3: "Not at all"
})
df["questionnaire_mode"] = df["questionnaire_mode"].replace({
    1: "Home Interview (18+ Yrs)",
    2: "ACASI (12 - 17 Yrs)"
})

In [ ]:
df["arm_selected"] = df["arm_selected"].replace({
    "L": "Left",
    "R": "Right",
    "": np.nan
})

In [ ]:
df = df.drop(columns=[
        "systolic_bp_1",	"diastolic_bp_1",	"systolic_bp_2",
        "diastolic_bp_2",	"systolic_bp_3",	"diastolic_bp_3",
        "pulse_1",	"pulse_2",	"pulse_3"
        ])

In [ ]:
df.head()

## Data Visualization and Analysis

### Visualization 1

In [ ]:
viz1 = df[
    df["bmi_computed"].notna() &
    df["cholesterol_mmol_l"].notna() &
    df["age_years"].notna() &
    df["gender"].notna() &
    df["smoking_status"].notna()
].copy()

viz1["age_years"] = viz1["age_years"].astype(int)

viz1 = viz1.sort_values("age_years")

In [ ]:
source = ColumnDataSource(viz1)
filtered_source = ColumnDataSource(viz1.copy())


p = figure(
    height=500,
    width=700,
    title="BMI vs Cholesterol (mmol/L)",
    x_axis_label="BMI (kg/m²)",
    y_axis_label="Total Cholesterol (mmol/L)",
    tools="pan,wheel_zoom,box_zoom,reset"
)

scatter = p.circle(
    source=filtered_source,
    x="bmi_computed",
    y="cholesterol_mmol_l",
    size=7,
    alpha=0.7,
    color="violet"
)


hover = HoverTool(
    renderers=[scatter],
    tooltips=[
        ("SEQN", "@SEQN"),
        ("Age", "@age_years"),
        ("Gender", "@gender"),
        ("Smoking status", "@smoking_status"),
        ("BMI", "@bmi_computed{0.0}"),
        ("Cholesterol (mmol/L)", "@cholesterol_mmol_l{0.00}"),
        ("Waist (cm)", "@waist_cm"),
        ("Hip (cm)", "@hip_cm"),
        ("SBP mean", "@bp_sys_mean"),
        ("DBP mean", "@bp_dias_mean")
    ]
)

p.add_tools(hover)

# Widgets
age_slider = Slider(start=int(viz1.age_years.min()),
                    end=int(viz1.age_years.max()),
                    value=int(viz1.age_years.max()),
                    step=1,
                    title="Maximum Age")

gender_select = Select(
    title="Gender",
    value="All",
    options=["All"] + sorted(viz1["gender"].unique().tolist())
)

smoke_select = Select(
    title="Smoking Status",
    value="All",
    options=["All"] + sorted(viz1["smoking_status"].unique().tolist())
)

#JS callback
callback = CustomJS(
    args=dict(
        source=source,
        filtered=filtered_source,
        age=age_slider,
        gender=gender_select,
        smoke=smoke_select
    ),
    code="""
    const data = source.data;
    const f = filtered.data;

    const age_max = age.value;
    const gender_val = gender.value;
    const smoke_val = smoke.value;

    const N = data['SEQN'].length;

    // Reset filtered source
    for (let key in f) {
        f[key] = [];
    }

    for (let i = 0; i < N; i++) {
        // Apply filtering conditions
        if (data['age_years'][i] <= age_max &&
            (gender_val === "All" || data['gender'][i] === gender_val) &&
            (smoke_val === "All" || data['smoking_status'][i] === smoke_val))
        {
            for (let key in data) {
                f[key].push(data[key][i]);
            }
        }
    }

    filtered.change.emit();
"""
)

# Linking callback to widgets
age_slider.js_on_change("value", callback)
gender_select.js_on_change("value", callback)
smoke_select.js_on_change("value", callback)


layout = column(
    p,
    row(age_slider, gender_select, smoke_select)
)

show(layout)

This visualization allows exploration of how body mass index (BMI) relates to total cholesterol levels while interactively filtering by age, gender, and smoking status.

Several patterns emerge:

* Cholesterol generally increases as BMI increases, especially past the normal BMI range. This aligns with metabolic risk literature showing adiposity-associated high "bad" cholesterol and low "good" cholesterol.

* Older participants tend to have higher cholesterol.
Sliding the age filter upward reveals a clear upward shift in cholesterol distribution.

* Gender differences become visible:

  * Males show slightly lower cholesterol variability.

  * Females cluster slightly more tightly in lower BMI ranges.

* Smoking status is highly informative:

  * Current smokers show elevated cholesterol but also more variability.

  * Former smokers occupy higher BMI and cholesterol combinations.

  * Never smokers have the healthiest overall profile.

* Hovering individual points reveals the interaction between BMI, waist/hip circumference, and blood pressure, offering a comprehensive cardiometabolic profile.

Overall, this plot demonstrates a strong, interpretable relationship between BMI and cholesterol, with demographic and behavioural factors contributing meaningfully.

### Visualization 2

In [ ]:
df_bp = df.copy()

# Only rows with valid blood pressure
df_bp = df_bp[df_bp["bp_sys_mean"].notna()]

source = ColumnDataSource(df_bp)
filtered_source = ColumnDataSource(df_bp.copy())

In [ ]:
counts, edges = np.histogram(df_bp["bp_sys_mean"], bins = 20)

hist_source = ColumnDataSource(data=dict(
    left=edges[:-1],
    right=edges[1:],
    top=counts
))

In [ ]:
p = figure(
    height=450, width=700,
    title="Distribution of Systolic Blood Pressure",
    tools="pan,wheel_zoom,box_zoom,reset,hover",
    x_axis_label="Systolic BP (mmHg)",
    y_axis_label="Count"
)

p.quad(
    top="top",
    bottom=0,
    left="left",
    right="right",
    source=hist_source,
    fill_color="royalblue",
    line_color="white",
    alpha=0.6
)

hover = p.select_one(HoverTool)
hover.tooltips = [
    ("Range", "@left{0}–@right{0}"),
    ("Count", "@top"),
]

age_slider = RangeSlider(
    start=int(df_bp.age_years.min()),
    end=int(df_bp.age_years.max()),
    value=(int(df_bp.age_years.min()), int(df_bp.age_years.max())),
    step=1,
    title="Age Range"
)

bmi_select = Select(
    title="BMI Category",
    value="All",
    options=["All"] + sorted(df_bp["bmi_category"].dropna().unique().tolist())
)

smoke_select = Select(
    title="Smoking Status",
    value="All",
    options=["All"] + sorted(df_bp["smoking_status"].dropna().unique().tolist())
)

# JS callback
callback = CustomJS(
    args=dict(
        source=source,
        filtered_source=filtered_source,
        hist_source=hist_source
    ),
    code="""
        const age_range = age_slider.value;
        const bmi_val = bmi_select.value;
        const smoke_val = smoke_select.value;

        const data = source.data;
        const fdata = filtered_source.data;

        const age = data['age_years'];
        const bmi = data['bmi_category'];
        const smoke = data['smoking_status'];
        const bp = data['bp_sys_mean'];

        // clear filtered data
        for (let key in fdata) {
            fdata[key] = [];
        }

        // filter rows
        for (let i = 0; i < bp.length; i++) {
            if (bp[i] == null) continue;

            const in_age = (age[i] >= age_range[0] && age[i] <= age_range[1]);
            const in_bmi = (bmi_val === "All" || bmi[i] === bmi_val);
            const in_smoke = (smoke_val === "All" || smoke[i] === smoke_val);

            if (in_age && in_bmi && in_smoke) {
                for (let key in fdata) {
                    fdata[key].push(data[key][i]);
                }
            }
        }

        // ---------------------------
        // Recompute histogram
        // ---------------------------
        const nbins = 20;
        const values = fdata['bp_sys_mean'];

        if (values.length > 0) {
            const min_val = Math.min(...values);
            const max_val = Math.max(...values);
            const bin_width = (max_val - min_val) / nbins;

            let left = [];
            let right = [];
            let top = new Array(nbins).fill(0);

            // Create bin edges
            for (let i = 0; i < nbins; i++) {
                left.push(min_val + i * bin_width);
                right.push(min_val + (i+1) * bin_width);
            }

            // Count values
            for (let v of values) {
                let idx = Math.floor((v - min_val) / bin_width);
                if (idx >= nbins) idx = nbins - 1;
                if (idx < 0) continue;
                top[idx]++;
            }

            hist_source.data['left'] = left;
            hist_source.data['right'] = right;
            hist_source.data['top'] = top;
        } else {
            // empty histogram
            hist_source.data['left'] = [];
            hist_source.data['right'] = [];
            hist_source.data['top'] = [];
        }

        filtered_source.change.emit();
        hist_source.change.emit();
    """
)

age_slider.js_on_change("value", callback)
bmi_select.js_on_change("value", callback)
smoke_select.js_on_change("value", callback)

# Adding widget references inside the callback
callback.args["age_slider"] = age_slider
callback.args["bmi_select"] = bmi_select
callback.args["smoke_select"] = smoke_select


layout = column(
    p,
    row(age_slider),
    row(bmi_select, smoke_select)
)

show(layout)

This visualization allows exploration of how systolic blood pressure (SBP) varies across the population while interactively filtering by age, BMI category, and smoking status.

Several patterns emerge:

* Systolic pressure increases with age. Sliding the age range upward reveals a clear upward shift in SBP values. Older adults exhibit both higher averages and greater spread, consistent with well-established research.

Clear differences appear across BMI categories:

* Obese participants show the highest SBP levels, reflecting the strong link between adiposity and hypertension.

* Overweight individuals cluster slightly lower but still above the healthy range.

* Normal weight participants display noticeably lower SBP values with with a smoother distribution due to large sample size.

* Underweight individuals, while fewer in number, generally show lower SBP but with slightly erratic spread due to small sample size.

Smoking status strongly influences SBP:

* Current smokers show elevated systolic pressure and greater variability even with a much smaller sample size, indicating higher cardiovascular strain.

* Former smokers seem to hover around current smokers. Could potentially show healthier SBP given a larger sample size for current smokers.

* Never smokers consistently exhibit the healthiest blood pressure profile, clustering around lower SBP ranges.

Hovering over individual bins highlights the ranges and counts of each bin.

Overall, this visualization illustrates a clear, interpretable relationship between systolic blood pressure and demographic/behavioural/metabolic factors, reinforcing known risk patterns and allowing efficient exploration of population subgroups.

### Visualization 3

In [ ]:
cols = [
    'SEQN', 'age_group', 'gender', 'bmi', 'bmi_category',
    'cholesterol_mg_dl', 'cholesterol_category',
    'smoking_status', 'hypertensive'
]

df_table = df[cols].copy()

source = ColumnDataSource(df_table)
filtered_source = ColumnDataSource(df_table.copy())

In [ ]:
search_box = TextInput(title="Search (ID, Age Group, Smoking Status, etc.)", value="")
chol_select = Select(
    title="Cholesterol Category Filter:",
    value="All",
    options=["All", "Desirable (<200)", "Borderline (200–239)", "High (≥240)"]
)

callback = CustomJS(
    args=dict(
        source=source,
        filtered_source=filtered_source,
        search=search_box,
        chol=chol_select
    ),
    code="""
    const data = source.data;
    const fdata = {};
    const search_val = search.value.toLowerCase();
    const chol_val = chol.value;

    // Prepare arrays
    for (const key in data) {
        fdata[key] = [];
    }

    for (let i = 0; i < data['SEQN'].length; i++) {
        let include = true;

        // --- Search filter ---
        if (search_val) {
            const row_str = (
                data['SEQN'][i] + " " +
                data['age_group'][i] + " " +
                data['gender'][i] + " " +
                data['bmi_category'][i] + " " +
                data['cholesterol_category'][i] + " " +
                data['smoking_status'][i]
            ).toLowerCase();

            if (!row_str.includes(search_val)) {
                include = false;
            }
        }

        // --- Cholesterol dropdown filter ---
        if (chol_val !== "All" && data['cholesterol_category'][i] !== chol_val) {
            include = false;
        }

        // Add matching rows
        if (include) {
            for (const key in data) {
                fdata[key].push(data[key][i]);
            }
        }
    }

    filtered_source.data = fdata;
"""
)

search_box.js_on_change("value", callback)
chol_select.js_on_change("value", callback)

# DataTable
columns = [
    TableColumn(field="SEQN", title="ID"),
    TableColumn(field="age_group", title="Age Group"),
    TableColumn(field="gender", title="Gender"),
    TableColumn(field="bmi", title="BMI"),
    TableColumn(field="bmi_category", title="BMI Category"),
    TableColumn(field="cholesterol_mg_dl", title="Cholesterol (mg/dL)"),
    TableColumn(field="cholesterol_category", title="Cholesterol Category"),
    TableColumn(field="smoking_status", title="Smoking Status"),
    TableColumn(field="hypertensive", title="Hypertensive")
]

data_table = DataTable(source=filtered_source, columns=columns, width=1000, height=400, styles={
        "background": "#f5f5f5",
        "color": "#000000",
        "font-size": "14px",
        "border": "1px solid #ccc"
    })

layout = column(
    row(search_box, chol_select),
    data_table
)

show(layout)

This interactive table provides a participant-level view of cardiovascular risk indicators, allowing detailed exploration beyond the scatter and trend visualizations. Filtering and searching reveal how cholesterol levels intersect with demographic and metabolic factors.

Several clear patterns are identifiable:

* Borderline levels of cholesterol and above clusters with obesity. Filtering for High (≥240 mg/dL) reveals that most individuals in this category belong to the Overweight or Obese BMI groups. Their BMI values frequently exceed the normal range, consistent with known metabolic risk profiles.

* Hypertension and excessive cholesterol often co-occur. Sorting or searching shows that many borderline and high-cholesterol participants are also flagged as hypertensive. This reinforces the link between high cholesterol and elevated cardiovascular risk.

Smoking status highlights behavioural risk patterns.

* Former smokers appear frequently in the Borderline and High cholesterol groups, likely reflecting cumulative exposure.

* Current smokers show variable cholesterol levels.

* Never smokers concentrate more in the Desirable (<200 mg/dL) range.

Age introduces a natural upward shift.
Older participants (especially 60+) dominate the higher cholesterol categories, consistent with age-related lipid metabolism changes.

The ability to search across multiple fields (ID, Age Group, Smoking Status, BMI category, etc.) allows targeted examination of subgroups such as hypertensive non-smoking females or obese males under 40—making this visualization a powerful quality-control and exploration tool. It complements the prior scatterplots by grounding high-level trends in the raw data.

### Visualization 4

In [ ]:
df_bubble = df.copy()

source = ColumnDataSource(df_bubble)
filtered_source = ColumnDataSource(df_bubble.copy())

# Dropdown filters
age_options = ["All"] + sorted(df_bubble["age_group"].dropna().unique().tolist())
smoke_options = ["All"] + sorted(df_bubble["smoking_status"].dropna().unique().tolist())

age_filter = Select(title="Age Group", value="All", options=age_options)
smoke_filter = Select(title="Smoking Status", value="All", options=smoke_options)

# Bubble colors based on BMI category
bmi_categories = ["Underweight", "Normal", "Overweight", "Obese"]

color_map = factor_cmap('bmi_category',
                        palette=["#4C9AFF", "#7ED957", "#FFC145", "#FF5C5C"],
                        factors=bmi_categories)

In [ ]:
p = figure(
    title="Waist vs Hip Circumference (Bubble = BMI)",
    width=850, height=550,
    x_axis_label="Waist circumference (cm)",
    y_axis_label="Hip circumference (cm)",
    tools="pan,wheel_zoom,box_zoom,reset,hover,save"
)

p.circle(
    x='waist_cm',
    y='hip_cm',
    size='bmi',          # mapping bubble size to BMI
    source=filtered_source,
    fill_color=color_map,
    fill_alpha=0.6,
    line_color="black",
    line_alpha=0.3
)

# Hover tooltips
p.hover.tooltips = [
    ("Waist (cm)", "@waist_cm"),
    ("Hip (cm)", "@hip_cm"),
    ("BMI", "@bmi{0.0}"),
    ("BMI Category", "@bmi_category"),
    ("Age", "@age_years"),
    ("Smoking Status", "@smoking_status"),
]

callback = CustomJS(args=dict(
    source=source,
    filtered=filtered_source,
    age_filter=age_filter,
    smoke_filter=smoke_filter
), code="""
    const data = source.data;
    const fdata = filtered.data;

    const age_val = age_filter.value;
    const smoke_val = smoke_filter.value;

    const N = data['waist_cm'].length;

    for (let col in fdata) {
        fdata[col] = [];
    }

    for (let i = 0; i < N; i++) {
        const age_match = (age_val === "All") || (data['age_group'][i] === age_val);
        const smoke_match = (smoke_val === "All") || (data['smoking_status'][i] === smoke_val);

        if (age_match && smoke_match) {
            for (let col in data) {
                fdata[col].push(data[col][i]);
            }
        }
    }

    filtered.change.emit();
""")

age_filter.js_on_change("value", callback)
smoke_filter.js_on_change("value", callback)

layout = column(row(age_filter, smoke_filter), p)
show(layout)

This visualization explores the relationship between waist circumference, hip circumference, and BMI using a bubble plot, with interactive filters for age group and smoking status. Bubble size represents BMI, while bubble color indicates BMI category, enabling quick identification of body composition patterns.

Several insights emerge:  
* A strong positive relationship exists between waist and hip circumference. As waist size increases, hip size tends to increase as well. This reflects general body proportionality and is consistent with anthropometric research.

BMI category shapes the structure of the plot:

* Obese individuals appear as the largest bubbles, clustering at higher waist and hip measurements.

* Overweight individuals occupy the mid-range with moderately sized bubbles.

* Normal-weight participants form a dense band at lower waist–hip combinations.

* Underweight participants, though few, appear as very small bubbles toward the lower end of both axes.

This produces a clear gradient where bubble size and color visually communicate metabolic risk.

Smoking status adds behavioural insight:

* Current smokers vaguely tend to show higher waist circumference relative to hip circumference, reflecting abdominal adiposity patterns documented among smokers.

* Former smokers show more variability, aligning with weight changes common after giving up smoking.

* Never smokers generally cluster in the healthier lower-risk regions of the plot, except for some outliers.

Age group filtering reveals additional structure:  
* Older adults show substantially higher waist and hip measurements, with bubbles shifting upward and rightward. Younger groups occupy the lower, tighter end of the distribution, indicating healthier anthropometric profiles.

Hovering over individual points highlights BMI, smoking behaviour, and age, giving a holistic view of metabolic and lifestyle factors.

Overall, this visualization provides an intuitive and interesting multidimensional look at body composition, revealing how waist, hip, BMI, age, and smoking behaviour interact to shape health risk.

### Visualization 5

In [ ]:
base_source = ColumnDataSource(df)

filtered_source = ColumnDataSource({col: base_source.data[col] for col in base_source.data})

age_select = Select(
    title="Select Age Group:",
    value="All",
    options=["All"] + sorted(df["age_group"].dropna().unique().astype(str).tolist())
)

In [ ]:
callback = CustomJS(args=dict(src=base_source, tgt=filtered_source, select=age_select), code="""
    const group = select.value;
    const data = src.data;
    const tdata = tgt.data;

    for (let k in data) { tdata[k] = []; }

    for (let i = 0; i < data['SEQN'].length; i++) {
        if (group === "All" || data['age_group'][i] === group) {
            for (let k in data) { tdata[k].push(data[k][i]); }
        }
    }

    tgt.change.emit();
""")

age_select.js_on_change("value", callback)

In [ ]:
# Tab A: Blood Pressure vs. Smoking
bp_fig = figure(
    x_range=sorted(df["smoking_status"].dropna().unique().tolist()),
    height=400, width=700,
    title="Systolic vs Diastolic Blood Pressure by Smoking Status"
)

bp_fig.circle(
    x=jitter("smoking_status", width=0.3, range=bp_fig.x_range),
    y="bp_sys_mean", source=filtered_source,
    size=7, alpha=0.4, color="firebrick", legend_label="Systolic"
)

bp_fig.circle(
    x=jitter("smoking_status", width=0.3, range=bp_fig.x_range),
    y="bp_dias_mean", source=filtered_source,
    size=7, alpha=0.4, color="yellow", legend_label="Diastolic"
)

bp_fig.legend.location = "top_right"
bp_panel = TabPanel(child=bp_fig, title="Blood Pressure")


# Tab B: Cholesterol vs. Smoking

chol_fig = figure(
    x_range=sorted(df["smoking_status"].dropna().unique().tolist()),
    height=400, width=700,
    title="Total Cholesterol by Smoking Status"
)

chol_fig.circle(
    x=jitter("smoking_status", width=0.3, range=chol_fig.x_range),
    y="cholesterol_mg_dl",
    source=filtered_source,
    size=7, alpha=0.4, color="green"
)

chol_panel = TabPanel(child=chol_fig, title="Cholesterol")


# Tab C: BMI vs. Smoking

bmi_fig = figure(
    x_range=sorted(df["smoking_status"].dropna().unique().tolist()),
    height=400, width=700,
    title="BMI Distribution by Smoking Status"
)

bmi_fig.circle(
    x=jitter("smoking_status", width=0.3, range=bmi_fig.x_range),
    y="bmi",
    source=filtered_source,
    size=7, alpha=0.4, color="purple"
)

bmi_panel = TabPanel(child=bmi_fig, title="BMI")



tabs = Tabs(tabs=[bp_panel, chol_panel, bmi_panel])

layout = column(age_select, tabs)
show(layout)

This tabbed visualization offers a focused comparison of cardiometabolic indicators across smoking status groups. Each tab isolates one dimension of cardiovascular health, making it easy to observe both distinct patterns and broader trends in how smoking relates to physiological risk.


The first tab plots systolic (SBP) and diastolic (DBP) blood pressure values using jittered points for each smoking group. Several developments emerge:

* Current smokers consistently occupy the upper range of SBP and DBP values and exhibit greater spread particularly in SBP, reflecting nicotine-associated increases in vascular resistance.

* Former smokers show partial improvement. Their SBP and DBP levels shift downward compared to current smokers but remain higher and more dispersed than those of never smokers.

* Never smokers cluster tightly in the healthiest range.

Together, these patterns highlight how smoking elevates the cardiovascular load and how giving up smoking leads to, but does not fully achieve, risk normalization.


The cholesterol tab reveals a similar gradient across groups.

* Current smokers seem to show slightly higher cholesterol levels on average, along with greater variability, aligning with known links between smoking and high cholesterol.

* Former smokers seem to again sit between the two extremes, improved mean relative to current smokers, yet still higher than never smokers.

* Never smokers seem to display the lowest mean of cholesterol values, indicating more favourable baseline lipid profiles.

This tab reinforces the metabolic strain introduced by smoking and the only partially reversible nature of lipid related risk factors.


The third tab contextualizes smoking behaviour against overall body composition.

* Current smokers, even accounting for its smaller sample size, has high variability and may have the highest mean BMI and greatest minimum out of the 3 categories of smokers.

* Former smokers seem to show a slight shift toward healthier ranges.

* Never smokers cluster around the healthiest BMI values on average.

Although BMI is influenced by many external factors, the visual difference across smoking groups highlights how smoking status intersects with weight related risk.


Across all three tabs, a strong narrative emerges:
Smoking status is strongly associated with less favourable cardiometabolic profiles. Current smokers show elevated blood pressure, higher cholesterol, and greater BMI variability, former smokers exhibit clear improvement but not complete normalization, and never smokers maintain the healthiest overall distributions.

## Data Privacy and Ethics Reflection

Although the analysis uses publicly available NHANES data, it is important to consider the privacy and ethical implications inherent in working with human health information. NHANES applies rigorous de-identification standards, yet the risk of re-identification can never be fully eliminated, particularly when datasets include detailed demographic variables such as age, race/ethnicity, income, and health measurements. Analyses must therefore avoid attempts to infer or reconstruct personal identities and should refrain from linking the dataset with external sources that could increase re-identification risk. Ethical practice also requires interpreting results responsibly, avoiding deterministic statements about individuals or demographic groups, and acknowledging the broader social, behavioural, and environmental factors that influence health outcomes. Throughout the project, care was taken to treat all variables as aggregated statistical patterns rather than personal attributes, and the findings are discussed in a way that prioritizes respect for participant confidentiality and avoids condemning vulnerable groups.

## Conclusion

This analysis demonstrates how integrated NHANES datasets can be used to reveal structure within population health patterns when combined with interactive visualization tools. Across the five visualizations, several consistent themes emerged. Higher BMI and increasing age were accompanied by less favourable cholesterol and blood pressure profiles, smoking status showed grouping across cardiometabolic measures, and demographic factors such as age and body composition influenced health indicators in interpretable ways. These interactive elements allowed deeper exploration of subgroups and made it possible to observe how risk factors cluster together rather than acting independently.

At the same time, the use of de-identified health data underscores the need for careful ethical handling. While NHANES follows strong privacy standards, responsible analysis requires avoiding re-identification risks, ensuring interpretations remain population-level, and presenting findings without reinforcing stigma or bias. Overall, the notebook illustrates how rigorous data wrangling, thoughtful visual design, and ethical awareness can work together to generate meaningful insights in public health analytics.

### Generative AI Assistance Acknowledgement

Portions of this notebook were prepared with support from generative AI tools (ChatGPT). Assistance was used primarily for helping design interactive Bokeh visualizations—particularly in constructing and simplifying the CustomJS callbacks. It was also used in refining explanations and improving clarity of written sections. All generated output was reviewed, validated, and finalized by the author.